In [1]:
%%capture
%pip install pycox lifelines scikit-survival optuna tqdm

In [2]:
import gc
import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
import torchtuples as tt
import xgboost as xgb
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
from pycox.models import CoxPH
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
from sksurv.datasets import load_breast_cancer
from sksurv.ensemble import RandomSurvivalForest
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored
from tqdm.auto import tqdm
from pycox.models import DeepHitSingle
from pycox.preprocessing.label_transforms import LabTransDiscreteTime
from sksurv.datasets import load_whas500
from lifelines.datasets import load_rossi
from sksurv.metrics import integrated_brier_score
from pycox.models import CoxTime
from pycox.preprocessing.label_transforms import LabTransCoxTime
import google

In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

df_colorectal = pd.read_csv('/content/drive/MyDrive/colorectal_data_cleaned.csv')

Mounted at /content/drive


In [4]:
df_colorectal.shape

(831287, 33)

In [5]:
from sklearn.preprocessing import MinMaxScaler

# scale continuous features to [0, 1]
continuous_features = ["Age recode with single ages and 90+", "Regional nodes positive (1988+)",
                    "Regional nodes examined (1988+)", "Tumor_Size"]
scaler = MinMaxScaler()
df_colorectal[continuous_features] = scaler.fit_transform(df_colorectal[continuous_features])

In [6]:
print(df_colorectal[continuous_features].describe().loc[['min', 'max']])

     Age recode with single ages and 90+  Regional nodes positive (1988+)  \
min                                  0.0                              0.0   
max                                  1.0                              1.0   

     Regional nodes examined (1988+)  Tumor_Size  
min                              0.0         0.0  
max                              1.0         1.0  


In [7]:
df_colorectal.head()

,Age recode with single ages and 90+,Regional nodes positive (1988+),Regional nodes examined (1988+),Tumor_Size,T,E,Grade_Ordinal,Stage_Ordinal,Is_Married,Sex_Male,...,Anatomy_Group_Right_Colon,Anatomy_Group_Unknown,Surgery_Group_No_Surgery,Surgery_Group_Partial_Colectomy,Surgery_Group_Radical_or_NOS,Surgery_Group_Total_Colectomy,Surgery_Group_Unknown,CEA_Result_20.0,CEA_Result_30.0,CEA_Result_Unknown
0,0.766667,0.000000,0.105263,0.040445,49.0,1,1.0,0.0,0,1.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
1,0.433333,0.010309,0.084211,0.035389,179.0,0,2.0,4.0,0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
2,0.911111,0.000000,0.000000,0.040445,0.0,1,3.0,0.0,0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,0.744444,0.000000,0.000000,0.050556,102.0,0,2.0,0.0,0,0.0,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,0.866667,0.020619,0.168421,0.070779,4.0,1,0.0,2.0,1,1.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0


In [8]:
# extract features and target variables
X_colorectal = df_colorectal.drop(columns=["T", "E"]).values.astype("float32")
T_colorectal = df_colorectal["T"].values.astype("float32")
E_colorectal = df_colorectal["E"].values.astype("float32")

In [9]:
X_train, X_test, T_train, T_test, E_train, E_test = train_test_split(
    X_colorectal, T_colorectal, E_colorectal, test_size=0.2, random_state=42, stratify=E_colorectal
)

In [ ]:
# cox proportional hazards
def cph(X_train, T_train, E_train, X_test, times):
    df_train = pd.DataFrame(X_train)
    df_train["time"] = T_train
    df_train["event"] = E_train

    cph = CoxPHFitter(penalizer=0.0)  # no penalisation

    try:
        cph.fit(df_train, duration_col="time", event_col="event")

        # predictions flattened to a 1D array
        preds = cph.predict_partial_hazard(pd.DataFrame(X_test)).values.flatten()

        # return 2d  array for IBS
        surv = cph.predict_survival_function(pd.DataFrame(X_test), times=times).values.T

        return preds, surv

    except Exception as e:
        print(f"CPH failed to converge: {e}")
        return None

In [ ]:
def cox_lasso(X_train, T_train, E_train, X_test, times=None):
    try:
        # scikit-survival requires a structured array for the target variable
        y_train = np.array([(bool(e), t) for e, t in zip(E_train, T_train)],
                           dtype=[('event', 'bool'), ('time', 'float32')])

        # Initialize Lasso (l1_ratio=1.0 is pure Lasso / L1 penalty)
        lasso_model = CoxnetSurvivalAnalysis(l1_ratio=1.0, fit_baseline_model=True)

        # fit model
        lasso_model.fit(X_train, y_train)

        # predict and flatten to ensure a clean 1D array for the bootstrap loop
        risk_scores = lasso_model.predict(X_test).flatten()

        # 2d array of survival functions for IBS
        surv_funcs = lasso_model.predict_survival_function(X_test)
        surv = np.array([fn(times) for fn in surv_funcs])

        return risk_scores, surv

    except Exception as e:
        print(f"Cox Lasso failed to converge: {e}")
        return None

In [12]:
# random survival forest
def rsf(X_train, T_train, E_train, X_test, times=None):
    try:
        y_train = np.array([(bool(e), t) for e, t in zip(E_train, T_train)], dtype=[("e", bool), ("t", float)])

        # fit rsf
        rsf_model = RandomSurvivalForest(n_estimators=100, min_samples_leaf=15, n_jobs=-1, random_state=0)
        rsf_model.fit(X_train, y_train)

        # risk for c-index
        risk = rsf_model.predict(X_test)

        # survival functions for IBS
        surv_funcs = rsf_model.predict_survival_function(X_test)
        surv = np.array([fn(times) for fn in surv_funcs])

        return risk, surv

    except Exception as e:
        print(f"RSF failed: {e}")
        return None, None if times is not None else None

In [13]:
def deepsurv(X_train, T_train, E_train, X_test, params, times):
    try:
        # Ensure arrays are float32 for PyTorch compatibility
        X_train = X_train.astype("float32")
        T_train = T_train.astype("float32")
        E_train = E_train.astype("float32")
        X_test = X_test.astype("float32")

        # Split strictly within the training set for Early Stopping
        X_tr, X_val, T_tr, T_val, E_tr, E_val = train_test_split(
            X_train, T_train, E_train, test_size=0.2, random_state=42, stratify=E_train
        )

        y_tr = (T_tr, E_tr)
        y_val = (T_val, E_val)

        in_features = X_tr.shape[1]
        out_features = 1
        num_nodes = [params["nodes"]] * params["layers"]
        dropout = params["dropout"]
        batch_norm = params.get("batch_norm", True)

        activation = nn.SELU if params.get("activation", "ReLU") == "SELU" else nn.ReLU

        net = tt.practical.MLPVanilla(
            in_features, num_nodes, out_features, batch_norm,
            dropout, activation=activation, output_bias=False
        )

        if params.get("optimiser", "adam") == "adam":
            optim = tt.optim.Adam(lr=params["lr"], weight_decay=params.get("l2", 0.0))
        elif params.get("optimiser") == "sgd":
            optim = tt.optim.SGD(lr=params["lr"], momentum=params.get("momentum", 0.9),
                                 weight_decay=params.get("l2", 0.0), nesterov=True)
        else:
            raise ValueError("Unknown optimiser")

        device = "cuda" if torch.cuda.is_available() else "cpu"
        model = CoxPH(net, optim, device=device)

        callbacks = [tt.callbacks.EarlyStopping(patience=15)]
        batch_size = params.get("batch_size", 64)

        model.fit(
            X_tr, y_tr,
            batch_size=batch_size,
            epochs=500,
            callbacks=callbacks,
            verbose=False,
            val_data=(X_val, y_val)
        )

        # Return the flat array of log-hazard (risk) scores
        risk = model.predict_net(X_test).flatten()

        # survival probabilities
        _ = model.compute_baseline_hazards(X_tr, y_tr)  # calculate baseline hazards
        surv_df = model.predict_surv_df(X_test)
        surv = surv_df.reindex(times, method='ffill').bfill().values.T

        return risk, surv

    except Exception as e:
        print(f"DeepSurv failed: {e}")
        return None

In [14]:
def optimise_deepsurv(X_train, T_train, E_train, n_trials=30):
    X_train, T_train, E_train = X_train.astype("float32"), T_train.astype("float32"), E_train.astype("float32")

    X_tr, X_val, T_tr, T_val, E_tr, E_val = train_test_split(
        X_train, T_train, E_train, test_size=0.2, random_state=42, stratify=E_train
    )

    def objective(trial):
        # parameters to optimise
        n_nodes = trial.suggest_int("nodes", 16, 128, step=16)
        n_layers = trial.suggest_int("layers", 1, 4)
        dropout = trial.suggest_float("dropout", 0.1, 0.6)
        lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        l2 = trial.suggest_float("l2", 1e-4, 1e-1, log=True)
        batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
        activation_name = trial.suggest_categorical("activation", ["ReLU", "SELU"])
        batch_norm = trial.suggest_categorical("batch_norm", [True, False])

        in_features = X_tr.shape[1]
        out_features = 1
        num_nodes = [n_nodes] * n_layers
        activation = nn.SELU if activation_name == "SELU" else nn.ReLU

        # deepsurv
        net = tt.practical.MLPVanilla(
            in_features, num_nodes, out_features,
            batch_norm=batch_norm, dropout=dropout,
            activation=activation, output_bias=False
        )

        optim = tt.optim.Adam(lr=lr, weight_decay=l2)
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model = CoxPH(net, optim, device=device)

        callbacks = [tt.callbacks.EarlyStopping(patience=15)]

        model.fit(
            X_tr, (T_tr, E_tr),
            batch_size=batch_size, epochs=500,
            callbacks=callbacks, verbose=False,
            val_data=(X_val, (T_val, E_val))
        )

        risk = model.predict_net(X_val).flatten()
        c_index = concordance_index(T_val, -risk, E_val)

        # memory cleanup
        del model, net, optim
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

        return c_index

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, n_jobs=1, show_progress_bar=True)

    return study.best_params

In [15]:
import torch.nn as nn

class CoxTimeMLP(nn.Module):
    """
    A custom wrapper to handle PyCox's dual-input format for Cox-Time.
    It catches (covariates, time) and concatenates them into a single tensor.
    """
    def __init__(self, in_features, num_nodes_list, out_features, batch_norm, dropout):
        super().__init__()
        self.mlp = tt.practical.MLPVanilla(
            in_features, num_nodes_list, out_features,
            batch_norm=batch_norm, dropout=dropout,
            activation=nn.ReLU, output_bias=False
        )

    def forward(self, x, time):
        # Concatenate features and time along the column dimension (dim=1)
        x_time = torch.cat([x, time], dim=1)
        return self.mlp(x_time)

In [ ]:
def coxtime(X_train, T_train, E_train, X_test, params, times):
    try:
        # Ensure float32 for PyTorch compatibility
        X_train = X_train.astype("float32")
        T_train = T_train.astype("float32")
        E_train = E_train.astype("float32")
        X_test = X_test.astype("float32")

        labtrans = LabTransCoxTime()
        y_train_cox = labtrans.fit_transform(T_train, E_train)

        X_tr, X_val, y_tr_time, y_val_time, y_tr_event, y_val_event = train_test_split(
            X_train, y_train_cox[0], y_train_cox[1],
            test_size=0.2, random_state=42, stratify=y_train_cox[1]
        )

        y_tr = (y_tr_time, y_tr_event)
        y_val = (y_val_time, y_val_event)

        # architecture update
        in_features = X_train.shape[1] + 1
        out_features = 1
        num_nodes_list = [params.get("nodes", 32)] * params.get("layers", 2)
        dropout = params.get("dropout", 0.2)

        # use the custom CoxTimeMLP to handle the dual-input format
        net = CoxTimeMLP(
            in_features=in_features,
            num_nodes_list=num_nodes_list,
            out_features=out_features,
            batch_norm=True,
            dropout=dropout
        )

        optim = tt.optim.Adam(lr=params.get("lr", 1e-3))
        device = "cuda" if torch.cuda.is_available() else "cpu"

        model = CoxTime(net, optim, labtrans=labtrans, device=device)
        callbacks = [tt.callbacks.EarlyStopping(patience=10)]

        # train model
        model.fit(
            X_tr, y_tr,
            batch_size=params.get("batch_size", 256),
            epochs=200,
            callbacks=callbacks,
            verbose=False,
            val_data=(X_val, y_val)
        )

        # predict
        model.compute_baseline_hazards(batch_size=2048)
        surv_df = model.predict_surv_df(X_test, batch_size=2048)

        # compute risk
        expected_survival_time = surv_df.sum().values
        risk = -expected_survival_time

        # compute survival probabilities
        surv = surv_df.reindex(times, method="ffill").bfill().values.T

        return risk, surv

    except Exception as e:
        print(f"Cox-Time evaluation failed: {e}")
        return None, None

In [ ]:
def optimise_coxtime(X_train, T_train, E_train, n_trials=15):
    # Ensure float32 for PyTorch compatibility
    X_train = X_train.astype("float32")
    T_train = T_train.astype("float32")
    E_train = E_train.astype("float32")

    def objective(trial):
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # parameters to optimise
        n_nodes = trial.suggest_int("nodes", 32, 128, step=32) # Increased upper limit
        n_layers = trial.suggest_int("layers", 1, 3)
        dropout = trial.suggest_float("dropout", 0.1, 0.5)
        lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])

        # transform the target variable for Cox-Time
        labtrans = LabTransCoxTime()
        y_train_cox = labtrans.fit_transform(T_train, E_train)

        # Splitting keeping track of the original T and E for validation
        X_tr, X_val, y_tr_time, y_val_time, y_tr_event, y_val_event, T_tr_orig, T_val_orig, E_tr_orig, E_val_orig = train_test_split(
            X_train, y_train_cox[0], y_train_cox[1], T_train, E_train,
            test_size=0.2, random_state=42, stratify=y_train_cox[1]
        )

        y_tr = (y_tr_time, y_tr_event)
        y_val = (y_val_time, y_val_event)

        # build the model architecture
        in_features = X_tr.shape[1] + 1

        # CoxTime always outputs exactly 1 node
        out_features = 1
        num_nodes_list = [n_nodes] * n_layers

        net = CoxTimeMLP(
            in_features=in_features,
            num_nodes_list=num_nodes_list,
            out_features=out_features,
            batch_norm=True,
            dropout=dropout
        )

        optim = tt.optim.Adam(lr=lr)
        device = "cuda" if torch.cuda.is_available() else "cpu"

        model = CoxTime(net, optim, labtrans=labtrans, device=device)
        callbacks = [tt.callbacks.EarlyStopping(patience=10)]

        # train the model
        model.fit(
            X_tr, y_tr,
            batch_size=batch_size, epochs=200,
            callbacks=callbacks, verbose=False,
            val_data=(X_val, y_val)
        )

        # compute baseline hazards
        model.compute_baseline_hazards(batch_size=2048)

        # predict the full survival dataframe for the validation set
        surv_df = model.predict_surv_df(X_val, batch_size=2048)

        # compute risk using expected survival time
        expected_survival_time = surv_df.sum().values
        risk = -expected_survival_time

        # compute c-index
        c_index = concordance_index(T_val_orig, risk, E_val_orig)

        # memory cleanup
        del model, net, optim, risk, surv_df
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return c_index

    # run study
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction="maximize")

    study.optimize(objective, n_trials=n_trials, n_jobs=1, show_progress_bar=True)

    return study.best_params

In [ ]:
n_trials = 5

In [19]:
# params_deepsurv = optimise_deepsurv(X_train, T_train, E_train, n_trials=n_trials)

In [ ]:
# print(params_deepsurv)

In [21]:
params_deepsurv = {'nodes': 32, 'layers': 4, 'dropout': 0.19174069597840984, 'lr': 0.00033160112259585973, 'l2': 0.0021939186946472143, 'batch_size': 128, 'activation': 'ReLU', 'batch_norm': True}

In [27]:
params_coxtime = optimise_coxtime(X_train, T_train, E_train, n_trials=n_trials)

Starting Cox-Time Hyperparameter Optimization...


  0%|          | 0/5 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torchtuples/tupletree.py:597: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  return self.tuple_.apply(lambda x: x[index])
/usr/local/lib/python3.12/dist-packages/torchtuples/tupletree.py:597: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  return self.tuple_.apply(lambda x: x[index]


Best Trial:
  C-Index: 0.2166
  Params: 
    nodes: 64
    layers: 2
    dropout: 0.45214348578933683
    lr: 0.0004820947400282524
    batch_size: 128


In [28]:
print(params_coxtime)

{'nodes': 64, 'layers': 2, 'dropout': 0.45214348578933683, 'lr': 0.0004820947400282524, 'batch_size': 128}


In [ ]:
params_coxtime = {'nodes': 64, 'layers': 2, 'dropout': 0.45214348578933683, 'lr': 0.0004820947400282524, 'batch_size': 128}

In [ ]:
import numpy as np
import pandas as pd
from sklearn.utils import resample
from lifelines.utils import concordance_index
from sksurv.metrics import integrated_brier_score
from tqdm import tqdm

def evaluate_seer(X_train, T_train, E_train,
                  X_test, T_test, E_test,
                  params_ds, params_dh,
                  n_bootstraps=1000):

    # ensure numpy arrays for indexing
    T_test = np.array(T_test)
    E_test = np.array(E_test)

    # format targets for IBS (sksurv)
    y_train_sksurv = np.array([(bool(e), t) for e, t in zip(E_train, T_train)], dtype=[("e", bool), ("t", float)])
    y_test_sksurv = np.array([(bool(e), t) for e, t in zip(E_test, T_test)], dtype=[("e", bool), ("t", float)])

    # define the time grid for IBS integration (5th to 95th percentile of test events)
    mask = E_test == 1
    event_times = T_test[mask]
    times = np.linspace(np.percentile(event_times, 5), np.percentile(event_times, 95), 100)

    # train all models and get predictions
    risk_cph, surv_cph = cph(X_train, T_train, E_train, X_test, times=times)
    risk_lasso, surv_lasso = cox_lasso(X_train, T_train, E_train, X_test, times=times)
    risk_ds, surv_ds = deepsurv(X_train, T_train, E_train, X_test, params_ds, times=times)
    risk_dh, surv_dh = coxtime(X_train, T_train, E_train, X_test, params_dh, times=times)
    risk_rsf, surv_rsf = rsf(X_train, T_train, E_train, X_test, times=times)

    # group predictions into a dictionary
    models = {
        "Cox Proportional Hazards": (risk_cph, surv_cph),
        "Cox Lasso": (risk_lasso, surv_lasso),
        "DeepSurv": (risk_ds, surv_ds),
        "Cox-Time": (risk_dh, surv_dh),
        "Random Survival Forest": (risk_rsf, surv_rsf)
    }

    # initialise a dictionary to hold bootstrap results
    boot_results = {name: {"cidx": [], "ibs": []} for name in models.keys()}

    # start bootstrapping
    for i in tqdm(range(n_bootstraps), desc="Bootstrapping SEER Colorectal Data"):

        # resample the test set indices with replacement
        indices = resample(np.arange(len(X_test)), replace=True, random_state=i)

        T_boot = T_test[indices]
        E_boot = E_test[indices]
        y_test_boot = y_test_sksurv[indices]

        # skip if no events are present
        if np.sum(E_boot) == 0:
            continue

        # create time mask to prevent sksurv from extrapolating
        time_mask = (times >= T_boot.min()) & (times < T_boot.max())
        times_boot = times[time_mask]

        if len(times_boot) < 2:
            continue

        # evaluate all models
        for name, (risk, surv) in models.items():
            # Concordance Index
            c_index = concordance_index(T_boot, -risk[indices], E_boot)
            boot_results[name]["cidx"].append(c_index)

            # Integrated Brier Score
            ibs = integrated_brier_score(y_train_sksurv, y_test_boot, surv[indices][:, time_mask], times_boot)
            boot_results[name]["ibs"].append(ibs)

    # output mean and 95% CI for each metric and model
    def get_metrics(boot_list):
        if not boot_list:
            return "NaN", "NaN"
        mean_val = round(np.mean(boot_list), 3)
        lower_ci = round(np.percentile(boot_list, 2.5), 3)
        upper_ci = round(np.percentile(boot_list, 97.5), 3)
        return f"{mean_val:.3f}", f"({lower_ci:.3f}, {upper_ci:.3f})"

    # build table
    final_rows = []
    for name in models.keys():
        mean_c, ci_c = get_metrics(boot_results[name]["cidx"])
        mean_ibs, ci_ibs = get_metrics(boot_results[name]["ibs"])

        final_rows.append({
            "Model": name,
            "C-Index (Mean)": mean_c,
            "C-Index (95% CI)": ci_c,
            "IBS (Mean)": mean_ibs,
            "IBS (95% CI)": ci_ibs
        })

    return pd.DataFrame(final_rows)

In [ ]:
results_df = evaluate_seer(
    X_train, T_train, E_train,
    X_test, T_test, E_test,
    params_ds=params_deepsurv,
    params_dh=params_coxtime,
    n_bootstraps=10
)

display(results_df)

/usr/local/lib/python3.12/dist-packages/torchtuples/tupletree.py:597: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  return self.tuple_.apply(lambda x: x[index])
Bootstrapping SEER Colorectal Data: 100%|██████████| 10/10 [03:04<00:00, 18.48s/it]


,Model,C-Index (Mean),C-Index (95% CI),IBS (Mean),IBS (95% CI)
0,Cox Proportional Hazards,0.749,"(0.747, 0.749)",0.167,"(0.166, 0.168)"
1,Cox Lasso,0.748,"(0.747, 0.749)",0.167,"(0.166, 0.168)"
2,DeepSurv,0.785,"(0.784, 0.786)",0.150,"(0.149, 0.151)"
3,Cox-Time,0.784,"(0.783, 0.785)",0.148,"(0.148, 0.149)"
4,Random Survival Forest,0.776,"(0.775, 0.778)",0.147,"(0.147, 0.148)"
